# 02 · Valorización, liquidez y propuesta de modelo · Grupo 31

**Despliegue de Soluciones Analíticas · MIAD** · Autor de este notebook: Julián Zapata

Este notebook parte de la exploración de `01_exploracion_bogota.ipynb` y la convierte en **evidencia para el reporte y el tablero**:

| Sección | Qué hace | Decisión del inversionista |
|---|---|---|
| 1–2 | Carga y preparación (mismos criterios del notebook 01) | – |
| 3 | Valorización real por zona y rango de valor | 1. ¿Dónde concentrar la búsqueda? |
| 4 | Liquidez (probabilidad de reventa) por zona | 1. ¿Dónde concentrar la búsqueda? |
| 5 | Rango esperado del precio por zona | 2. ¿El precio de una oferta está en rango? |
| 6 | **Modelos supervisados** frente a su línea base, con validación temporal: rango del valor (principal) y reventa a 3 años (secundario) | 2. (lo que servirá la API) |
| 7 | Cifras clave para el reporte | – |

Todas las figuras se guardan en `reports/figures/` para usarlas directamente en el reporte.

## 1. Configuración

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = "../data"
FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

CSV_COMPRAVENTAS = "compraventas_0125_bogota.csv"
ARCHIVO_IPC = "ipc.csv"
ZONAS = {"50N": "Norte", "50C": "Centro", "50S": "Sur"}
ORDEN_ZONAS = ["Norte", "Centro", "Sur"]
COLORES = {"Norte": "#2a6f97", "Centro": "#e07a5f", "Sur": "#3d9970"}

# Mismos filtros del notebook 01 (se justifican allí)
DIAS_MIN = 30
VALOR_MIN = 5e6
RET_MIN, RET_MAX = 0.2, 5
DIAS_ANUALIZAR = 180
DIAS_ATIPICO_IDENTICO = 365
N_QUINTILES = 5
N_MIN_SEG = 100
BOOTSTRAP = 1000
SEMILLA = 42

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .3, "font.size": 10})

def guardar(fig, nombre):
    ruta = os.path.join(FIG_DIR, nombre)
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Figura guardada:", ruta)

## 2. Datos y preparación

Se repite la preparación del notebook 01 (tres formatos de fecha, llave del predio `orip + matrícula`, valor válido) para que este notebook sea autocontenido.

In [ ]:
cv = pd.read_csv(os.path.join(DATA_DIR, CSV_COMPRAVENTAS), dtype=str)
print(f"Compraventas leídas: {len(cv):,}")

cv["valor"] = pd.to_numeric(cv["valor"], errors="coerce")
cv["year_radica"] = pd.to_numeric(cv["year_radica"], errors="coerce").astype("Int64")
txt = cv["fecha_radica_texto"].str.strip()
cv["fecha"] = pd.Series(pd.NaT, index=cv.index, dtype="datetime64[ns]")
for n_car, fmt in {19: "%Y-%m-%d %H:%M:%S", 10: "%d/%m/%Y", 8: "%d/%m/%y"}.items():
    m = txt.str.len() == n_car
    cv.loc[m, "fecha"] = pd.to_datetime(txt[m], format=fmt, errors="coerce")

cv["orip"] = cv["orip"].str.strip()
cv["zona"] = cv["orip"].map(ZONAS)
cv["predio"] = pd.factorize(cv["orip"] + "-" + cv["matricula"].str.strip())[0]
cv["valor_valido"] = cv["valor"] > 0
cv = cv.dropna(subset=["fecha", "zona"]).copy()
print(f"Con fecha y zona de Bogotá: {len(cv):,}  |  con valor: {cv['valor_valido'].mean():.1%}")

# IPC (índice mensual, base dic-2018 = 100)
ipc = pd.read_csv(os.path.join(DATA_DIR, ARCHIVO_IPC), sep=None, engine="python", encoding="utf-8-sig")
ipc.columns = [c.strip().lower() for c in ipc.columns]
ipc["fecha"] = pd.to_datetime(ipc["fecha"])
ipc["indice"] = pd.to_numeric(ipc["indice"].astype(str).str.replace(",", "."), errors="coerce")
indice = ipc.assign(mes=ipc["fecha"].dt.to_period("M")).set_index("mes")["indice"]
IPC_REF = indice.dropna().iloc[-1]
MES_REF = indice.dropna().index[-1]
print(f"IPC: {indice.index.min()} a {MES_REF}  |  valores en pesos constantes de {MES_REF}")

## 3. Valorización real por zona y rango de valor

Pares de ventas repetidas del mismo predio, con los filtros de calidad del notebook 01. El retorno real descuenta el IPC: `ret_real = (valor/IPC) / (valor_ant/IPC_ant)`.

In [ ]:
cvv = cv[cv["valor_valido"]].sort_values(["predio", "fecha", "pk"]).copy()
g = cvv.groupby("predio")
cvv["valor_ant"], cvv["fecha_ant"] = g["valor"].shift(), g["fecha"].shift()
p = cvv.dropna(subset=["valor_ant"]).copy()
p["dias"] = (p["fecha"] - p["fecha_ant"]).dt.days
p["ret_total"] = p["valor"] / p["valor_ant"]
p = p[(p["dias"] >= DIAS_ANUALIZAR) & (p["valor_ant"] >= VALOR_MIN) & (p["valor"] >= VALOR_MIN)
      & p["ret_total"].between(RET_MIN, RET_MAX)]
p = p[~((p["valor"] == p["valor_ant"]) & (p["dias"] <= DIAS_ATIPICO_IDENTICO))].copy()

p["ipc"] = p["fecha"].dt.to_period("M").map(indice)
p["ipc_ant"] = p["fecha_ant"].dt.to_period("M").map(indice)
p = p.dropna(subset=["ipc", "ipc_ant"])
p["ret_nom_anual"] = p["ret_total"] ** (365.25 / p["dias"]) - 1
p["ret_real_anual"] = ((p["valor"] / p["ipc"]) / (p["valor_ant"] / p["ipc_ant"])) ** (365.25 / p["dias"]) - 1
p["q"] = pd.qcut(p["valor_ant"], N_QUINTILES, labels=False, duplicates="drop") + 1
limites_q = p.groupby("q")["valor_ant"].agg(["min", "max"]) / 1e6
print(f"Pares en el cálculo principal: {len(p):,}")
print(f"Mediana nominal: {p['ret_nom_anual'].median():.2%}  |  mediana real: {p['ret_real_anual'].median():.2%}  |  "
      f"pares con retorno real negativo: {(p['ret_real_anual'] < 0).mean():.1%}")
print("\nRangos de valor (quintiles del valor de la primera venta, millones de pesos corrientes):")
display(limites_q.round(0))

### 3.1 Nominal frente a real
La inflación explica buena parte de lo que parece valorización.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(-0.6, 1.0, 65)
ax.hist(p["ret_nom_anual"].clip(-0.6, 1.0), bins=bins, alpha=.55, label="Nominal", color="#9aa5b1")
ax.hist(p["ret_real_anual"].clip(-0.6, 1.0), bins=bins, alpha=.65, label="Real (descontando IPC)", color="#2a6f97")
for s, c in [("ret_nom_anual", "#5f6b76"), ("ret_real_anual", "#1d4e6b")]:
    ax.axvline(p[s].median(), color=c, ls="--", lw=1.5)
ax.axvline(0, color="black", lw=.8)
ax.set_xlabel("Retorno anualizado"); ax.set_ylabel("Pares de ventas")
ax.set_title(f"Valorización anual de predios revendidos en Bogotá (n = {len(p):,})\n"
             f"Mediana nominal {p['ret_nom_anual'].median():.1%} vs. real {p['ret_real_anual'].median():.1%}")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}")); ax.legend()
guardar(fig, "f1_nominal_vs_real.png"); plt.show()

### 3.2 Mapa de calor: zona × rango de valor
Cada celda es la mediana del retorno real anual del segmento; entre paréntesis, el número de pares.

In [ ]:
rng = np.random.default_rng(SEMILLA)
def ic_mediana(x, B=BOOTSTRAP, nivel=.90):
    x = np.asarray(x, float)
    if len(x) < 2: return (np.nan, np.nan)
    meds = np.median(x[rng.integers(0, len(x), (B, len(x)))], axis=1)
    return tuple(np.quantile(meds, [(1 - nivel) / 2, 1 - (1 - nivel) / 2]))

filas = []
for (z, q), gq in p.groupby(["zona", "q"]):
    lo, hi = ic_mediana(gq["ret_real_anual"])
    filas.append({"zona": z, "q": int(q), "n": len(gq), "mediana": gq["ret_real_anual"].median(), "ic_inf": lo, "ic_sup": hi})
seg = pd.DataFrame(filas)
seg["suficiente"] = seg["n"] >= N_MIN_SEG

mat = seg.pivot(index="zona", columns="q", values="mediana").loc[ORDEN_ZONAS]
nn = seg.pivot(index="zona", columns="q", values="n").loc[ORDEN_ZONAS]
lim = np.nanmax(np.abs(mat.values))
fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(mat.values, cmap="RdYlGn", vmin=-lim, vmax=lim, aspect="auto")
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v, n = mat.values[i, j], nn.values[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:+.1%}\n(n={int(n)})" + ("" if n >= N_MIN_SEG else "*"), ha="center", va="center", fontsize=9)
ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels([f"Q{q}\n{limites_q.loc[q,'min']:.0f}–{limites_q.loc[q,'max']:.0f} M$" for q in mat.columns], fontsize=8)
ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels(mat.index); ax.grid(False)
ax.set_title("Mediana del retorno REAL anual por zona registral y rango de valor\n(* = menos de 100 pares, interpretar con cautela)")
fig.colorbar(im, ax=ax, format=plt.FuncFormatter(lambda x, _: f"{x:.0%}"), shrink=.85)
guardar(fig, "f2_heatmap_zona_rango.png"); plt.show()
display(seg.round(4))

### 3.3 Incertidumbre por segmento (intervalo del 90 % por bootstrap)

In [ ]:
s = seg.sort_values(["zona", "q"]).copy()
s["etq"] = s["zona"] + " · Q" + s["q"].astype(str)
s = s.set_index("zona").loc[ORDEN_ZONAS].reset_index()
fig, ax = plt.subplots(figsize=(8, 5.5))
y = np.arange(len(s))[::-1]
for yi, (_, r) in zip(y, s.iterrows()):
    c = COLORES[r["zona"]]
    ax.plot([r["ic_inf"], r["ic_sup"]], [yi, yi], color=c, lw=2.5, alpha=.8 if r["suficiente"] else .35)
    ax.plot(r["mediana"], yi, "o", color=c, ms=6, mfc=c if r["suficiente"] else "white")
ax.axvline(0, color="black", lw=.8)
ax.set_yticks(y); ax.set_yticklabels(s["etq"])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_xlabel("Retorno real anual (mediana e intervalo del 90 %)")
ax.set_title("¿Qué segmentos se valorizan por encima de la inflación?\n(punto hueco = menos de 100 pares)")
guardar(fig, "f3_ic_segmentos.png"); plt.show()

## 4. Liquidez: probabilidad de reventa por zona

No necesita el valor, así que usa los nueve años (2015–2023). Los predios que aún no se revenden se tratan como **censurados** (Kaplan-Meier).

In [ ]:
from lifelines import KaplanMeierFitter
FECHA_FIN = cv["fecha"].max()
liq = cv.sort_values(["predio", "fecha", "pk"])[["predio", "fecha", "zona", "year_radica", "pk"]].copy()
liq["fecha_sig"] = liq.groupby("predio")["fecha"].shift(-1)
liq["evento"] = liq["fecha_sig"].notna()
liq["dur"] = (liq["fecha_sig"].fillna(FECHA_FIN) - liq["fecha"]).dt.days
liq = liq[~(liq["evento"] & (liq["dur"] < DIAS_MIN))]
liq = liq[liq["dur"] > 0].copy()
print(f"Compraventas en el análisis: {len(liq):,}  |  con reventa observada: {liq['evento'].mean():.1%}")

def prob_reventa(d, anios):
    k = KaplanMeierFitter().fit(d["dur"] / 365.25, d["evento"])
    return 1 - float(k.predict(anios))

fig, ax = plt.subplots(figsize=(8, 4.5))
tabla_liq = {}
for z in ORDEN_ZONAS:
    d = liq[liq["zona"] == z]
    k = KaplanMeierFitter().fit(d["dur"] / 365.25, d["evento"])
    sf = 1 - k.survival_function_.iloc[:, 0]
    ax.step(sf.index, sf.values, where="post", color=COLORES[z], lw=2, label=f"{z} (n={len(d):,})")
    tabla_liq[z] = {f"P(reventa ≤ {a} años)": prob_reventa(d, a) for a in (1, 3, 5)}
ax.set_xlim(0, 8.5); ax.set_xlabel("Años desde la compra"); ax.set_ylabel("Probabilidad acumulada de reventa")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_title("Liquidez por zona registral de Bogotá (Kaplan-Meier, 2015–2023)"); ax.legend()
guardar(fig, "f4_liquidez_km.png"); plt.show()
tabla_liq = pd.DataFrame(tabla_liq).T
display(tabla_liq.style.format("{:.1%}"))

### 4.1 ¿La liquidez es estable en el tiempo? (validación retrospectiva)
Se compara la probabilidad de reventa a 3 años de las compras de 2015–2017 con las de 2018–2020.

In [ ]:
liq["cohorte"] = np.where(liq["year_radica"] <= 2017, "Compras 2015–2017",
                  np.where(liq["year_radica"] <= 2020, "Compras 2018–2020", "Compras 2021–2023"))
coh = pd.DataFrame({c: {z: prob_reventa(liq[(liq["cohorte"] == c) & (liq["zona"] == z)], 3) for z in ORDEN_ZONAS}
                    for c in ["Compras 2015–2017", "Compras 2018–2020"]})
fig, ax = plt.subplots(figsize=(7, 3.8))
coh.plot(kind="bar", ax=ax, color=["#9aa5b1", "#2a6f97"], rot=0, width=.7)
for cont in ax.containers:
    ax.bar_label(cont, labels=[f"{v:.1%}" for v in cont.datavalues], fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_ylabel("P(reventa ≤ 3 años)"); ax.set_xlabel("")
ax.set_title("Probabilidad de reventa a 3 años por zona y cohorte de compra")
guardar(fig, "f5_liquidez_cohortes.png"); plt.show()
display(coh.style.format("{:.1%}"))

## 5. Rango esperado del precio por zona (pesos constantes)

Referencia para la decisión 2. El segmento usa solo variables **ajenas al precio** (zona y año) para que la pregunta «¿este precio está en rango?» no sea circular.

In [ ]:
bv = cv[cv["valor_valido"] & (cv["valor"] >= VALOR_MIN)].copy()
bv["mes"] = bv["fecha"].dt.to_period("M")
bv["ipc"] = bv["mes"].map(indice)
bv = bv.dropna(subset=["ipc"])
bv["valor_real"] = bv["valor"] * IPC_REF / bv["ipc"]
rango = (bv.groupby(["zona", "year_radica"])["valor_real"].quantile([.10, .25, .50, .75, .90])
           .unstack() / 1e6)
rango["n"] = bv.groupby(["zona", "year_radica"]).size()

fig, ax = plt.subplots(figsize=(8, 4.2))
anios = sorted(bv["year_radica"].dropna().unique())
ancho = .25
for i, z in enumerate(ORDEN_ZONAS):
    r = rango.loc[z]
    x = np.arange(len(r)) + (i - 1) * ancho
    ax.vlines(x, r[0.10], r[0.90], color=COLORES[z], lw=2, alpha=.5)
    ax.vlines(x, r[0.25], r[0.75], color=COLORES[z], lw=7, label=z)
    ax.plot(x, r[0.50], "_", color="black", ms=12, mew=2)
ax.set_xticks(np.arange(len(anios))); ax.set_xticklabels([str(a) for a in rango.loc[ORDEN_ZONAS[0]].index])
ax.set_ylabel(f"Millones de pesos de {MES_REF}")
ax.set_title("Rango del valor de compraventa por zona y año\n(barra gruesa p25–p75 · línea p10–p90 · raya = mediana)")
ax.legend()
guardar(fig, "f6_rango_valor_zona.png"); plt.show()
display(rango.round(1))

## 6. Propuesta de modelo y baseline

### 6.1 Qué se propone
| Elemento | Definición |
|---|---|
| **Pregunta que responde** | Decisión 2: *dado un predio en una zona y una fecha, ¿el precio ofrecido está por debajo, dentro o por encima del rango esperado?* |
| **Tipo de modelo** | Modelo principal del reporte: **regresión del valor registrado**, en versión por cuantiles para predecir p10, p50 y p90 (pesos constantes) |
| **Variable objetivo** | `log(valor_real)` de cada compraventa con valor (2021–2023) |
| **Variables de entrada (v1)** | Zona registral, tiempo (meses desde ene-2021) y mes del año. *Limitación: el registro no trae área, estrato ni tipo de inmueble* |
| **Validación** | **Temporal**: entrenar con 2021–2022 y evaluar con 2023 (simula usar el pasado para juzgar ofertas futuras) |
| **Métricas** | Cobertura del intervalo p10–p90 (ideal ≈ 80 %), ancho del intervalo, *pinball loss* por cuantil |
| **Uso en el producto** | La API recibe zona + fecha + precio ofrecido y devuelve el rango y la clasificación; el tablero la muestra junto con la valorización y la liquidez de la zona |

Se comparan dos opciones, como pide el criterio de éxito del reporte (superar la línea base en validación temporal): **(A) línea base: percentiles por zona del año anterior** y **(B) Gradient Boosting por cuantiles** (equivalente a LightGBM en scikit-learn).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_pinball_loss

m = bv[bv["year_radica"] >= 2021].copy()
m["y"] = np.log(m["valor_real"])
m["t"] = (m["fecha"].dt.year - 2021) * 12 + m["fecha"].dt.month - 1
m["mes_anio"] = m["fecha"].dt.month
m["zona_cod"] = m["zona"].map({z: i for i, z in enumerate(ORDEN_ZONAS)})
train, test = m[m["year_radica"] <= 2022], m[m["year_radica"] == 2023]
print(f"Entrenamiento (2021–2022): {len(train):,}  |  Prueba (2023): {len(test):,}")
CUANTILES = [0.10, 0.50, 0.90]
X_COLS = ["zona_cod", "t", "mes_anio"]

# (A) Línea base del reporte: percentiles por segmento (zona) del año más reciente de entrenamiento (2022)
base = train[train["year_radica"] == 2022].groupby("zona")["y"].quantile(CUANTILES).unstack()
pred_A = {q: test["zona"].map(base[q]).values for q in CUANTILES}

# (B) Gradient Boosting por cuantiles
pred_B, modelos = {}, {}
for q in CUANTILES:
    mod = HistGradientBoostingRegressor(loss="quantile", quantile=q, max_iter=200, learning_rate=.05,
                                        categorical_features=[0], random_state=SEMILLA)
    mod.fit(train[X_COLS], train["y"])
    modelos[q] = mod
    pred_B[q] = mod.predict(test[X_COLS])

def evaluar(pred, nombre):
    y = test["y"].values
    cob = np.mean((y >= pred[0.10]) & (y <= pred[0.90]))
    ancho = np.median(np.exp(pred[0.90]) - np.exp(pred[0.10])) / 1e6
    return {"modelo": nombre, "cobertura p10–p90": cob, "ancho mediano del rango (M$)": ancho,
            **{f"pinball q{int(q*100)}": mean_pinball_loss(y, pred[q], alpha=q) for q in CUANTILES}}

res = pd.DataFrame([evaluar(pred_A, "A · Línea base (zona, año anterior)"), evaluar(pred_B, "B · Gradient Boosting cuantiles")])
display(res.round(3))

In [ ]:
# Figura: valores reales de 2023 frente al rango predicho, por zona
test = test.assign(p10=np.exp(pred_B[0.10]) / 1e6, p50=np.exp(pred_B[0.50]) / 1e6, p90=np.exp(pred_B[0.90]) / 1e6,
                   real=test["valor_real"] / 1e6, mes=test["fecha"].dt.to_period("M"))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, z in zip(axes, ORDEN_ZONAS):
    d = test[test["zona"] == z].groupby("mes").agg(real_p50=("real", "median"), real_p10=("real", lambda s: s.quantile(.1)),
                                                     real_p90=("real", lambda s: s.quantile(.9)),
                                                     p10=("p10", "mean"), p50=("p50", "mean"), p90=("p90", "mean"))
    x = d.index.to_timestamp()
    ax.fill_between(x, d["p10"], d["p90"], color=COLORES[z], alpha=.2, label="Rango predicho p10–p90")
    ax.plot(x, d["p50"], color=COLORES[z], lw=2, label="Mediana predicha")
    ax.plot(x, d["real_p50"], "k.--", lw=1, label="Mediana real")
    ax.plot(x, d["real_p10"], "k:", lw=.8); ax.plot(x, d["real_p90"], "k:", lw=.8, label="p10 / p90 reales")
    ax.set_title(z); ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel(f"Millones de pesos de {MES_REF}"); axes[0].legend(fontsize=7)
fig.suptitle("Validación temporal del modelo de rango de precio: predicción con 2021–2022 frente a lo observado en 2023")
plt.tight_layout(); guardar(fig, "f7_validacion_modelo.png"); plt.show()

# Ejemplo de uso (lo que haría la API)
def clasificar_oferta(zona, fecha, precio):
    f = pd.Timestamp(fecha)
    x = pd.DataFrame({"zona_cod": [ORDEN_ZONAS.index(zona)], "t": [(f.year - 2021) * 12 + f.month - 1], "mes_anio": [f.month]})
    lo, hi = [np.exp(modelos[q].predict(x))[0] for q in (0.10, 0.90)]
    estado = "por debajo del rango" if precio < lo else ("por encima del rango" if precio > hi else "dentro del rango")
    return f"{zona}, {f:%Y-%m}: rango esperado {lo/1e6:,.0f}–{hi/1e6:,.0f} M$ → oferta de {precio/1e6:,.0f} M$ está {estado}"
print(clasificar_oferta("Norte", "2023-06-01", 450e6))

### 6.2 Modelo secundario: probabilidad de reventa a 3 años (clasificación)
Complementa la liquidez por zona. Se usan compras con **3 años completos de observación** (hasta la fecha final del registro menos 3 años), así la etiqueta no está censurada.
- **Objetivo:** 1 si el predio se revende en 3 años o menos, 0 si no.
- **Variables:** zona, mes del año, tipo de predio y número de compraventas previas del predio desde 2015.
- **Validación temporal:** entrenar con compras de 2015–2017 y evaluar con compras de 2018–2020.
- **Línea base:** tasa de reventa por zona del periodo de entrenamiento.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve

r3 = cv.sort_values(["predio", "fecha", "pk"])[["predio", "fecha", "zona", "year_radica", "tipo_predio_zona", "pk"]].copy()
r3["n_previas"] = r3.groupby("predio").cumcount()
r3 = r3.join(liq[["dur", "evento"]], how="inner")             # misma definición de reventa de la sección 4
r3 = r3[r3["fecha"] <= FECHA_FIN - pd.Timedelta(days=1096)].copy()
r3["y"] = (r3["evento"] & (r3["dur"] <= 1096)).astype(int)
r3["zona_cod"] = r3["zona"].map({z: i for i, z in enumerate(ORDEN_ZONAS)})
r3["mes_anio"] = r3["fecha"].dt.month
r3["tipo_cod"] = pd.factorize(r3["tipo_predio_zona"].fillna("NA"))[0]
X3 = ["zona_cod", "mes_anio", "tipo_cod", "n_previas"]
tr3, te3 = r3[r3["year_radica"] <= 2017], r3[(r3["year_radica"] >= 2018) & (r3["year_radica"] <= 2020)]
print(f"Entrenamiento (compras 2015–2017): {len(tr3):,}  |  prueba (compras 2018–2020): {len(te3):,}  |  "
      f"tasa de reventa a 3 años: {tr3['y'].mean():.1%} / {te3['y'].mean():.1%}")

p_base = te3["zona"].map(tr3.groupby("zona")["y"].mean()).values
clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=.05, categorical_features=[0, 2], random_state=SEMILLA)
clf.fit(tr3[X3], tr3["y"])
p_mod = clf.predict_proba(te3[X3])[:, 1]
res3 = pd.DataFrame([
    {"modelo": "A · Línea base (tasa por zona)", "AUC": roc_auc_score(te3["y"], p_base), "Brier": brier_score_loss(te3["y"], p_base)},
    {"modelo": "B · Gradient Boosting", "AUC": roc_auc_score(te3["y"], p_mod), "Brier": brier_score_loss(te3["y"], p_mod)}])
display(res3.round(4))

fig, ax = plt.subplots(figsize=(5, 4.5))
for pr_, nom, c in [(p_base, "A · Línea base", "#9aa5b1"), (p_mod, "B · Gradient Boosting", "#2a6f97")]:
    fpr, tpr, _ = roc_curve(te3["y"], pr_)
    ax.plot(fpr, tpr, color=c, lw=2, label=f"{nom} (AUC={roc_auc_score(te3['y'], pr_):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=.8)
ax.set_xlabel("Tasa de falsos positivos"); ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("Reventa a 3 años: validación temporal\n(entrena 2015–2017, evalúa 2018–2020)"); ax.legend(fontsize=8)
guardar(fig, "f8_reventa_3anios_roc.png"); plt.show()

### 6.3 Lectura y siguientes pasos
- La cobertura cercana al 80 % indica si el rango está **bien calibrado**; el ancho indica qué tan **útil** es para el inversionista (un rango muy ancho es correcto pero poco informativo).
- Con solo zona y tiempo, el modelo B mejora poco frente al baseline: **la principal limitación es la falta de atributos del predio** (área, estrato, tipo). Para la Entrega 2 se evaluará enriquecer el registro (p. ej., catastro de Bogotá o variables a nivel de zona) y, para liquidez, un modelo de supervivencia con covariables (Cox).
- Este modelo es el que se empaquetará y servirá vía API; la valorización (sección 3) y la liquidez (sección 4) alimentan las vistas descriptivas del tablero.

## 7. Cifras clave para el reporte

In [ ]:
mejor = seg[seg["suficiente"]].sort_values("mediana", ascending=False).iloc[0]
peor = seg[seg["suficiente"]].sort_values("mediana").iloc[0]
cifras = pd.DataFrame([
    ("Pares de ventas repetidas (cálculo principal)", f"{len(p):,}"),
    ("Valorización nominal anual (mediana)", f"{p['ret_nom_anual'].median():.1%}"),
    ("Valorización real anual (mediana)", f"{p['ret_real_anual'].median():.1%}"),
    ("Pares con retorno real negativo", f"{(p['ret_real_anual'] < 0).mean():.1%}"),
    ("Segmento con mayor valorización real (n≥100)", f"{mejor['zona']} Q{mejor['q']}: {mejor['mediana']:.1%}"),
    ("Segmento con menor valorización real (n≥100)", f"{peor['zona']} Q{peor['q']}: {peor['mediana']:.1%}"),
    *[(f"P(reventa ≤ 3 años) · {z}", f"{tabla_liq.loc[z, 'P(reventa ≤ 3 años)']:.1%}") for z in ORDEN_ZONAS],
    ("Modelo B · cobertura p10–p90 en 2023", f"{res.loc[1, 'cobertura p10–p90']:.1%}"),
    ("Modelo B · ancho mediano del rango", f"{res.loc[1, 'ancho mediano del rango (M$)']:,.0f} M$"),
    ("Reventa a 3 años · AUC línea base / modelo", f"{res3.loc[0, 'AUC']:.3f} / {res3.loc[1, 'AUC']:.3f}"),
], columns=["cifra", "valor"])
display(cifras)
cifras.to_csv(os.path.join(FIG_DIR, "cifras_clave_02.csv"), index=False)